In [ ]:
from __future__ import annotations

import ast
import json
import re
from pathlib import Path
from statistics import mean
from typing import Final, TypedDict

from anthropic.types import Message, MessageParam, TextBlock

from learning_journey.claude.client import get_client

In [ ]:
class TestOut(TypedDict):
    output: str
    test_case: dict[str, str]
    score: int
    reasoning: str

In [ ]:
# Starting prompt (version 1)

# An f-string is evaluated the moment Python reaches that line, so it looks up task right then.
# Since task doesn't exist yet, you get a NameError. .format() never gets a chance to run.
PROMPT_TEMPLATE: Final[str] = """
Please provide a solution to the following task:
{task}
"""

EVAL_PROMPT_TEMPLATE: Final[str] = """
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {task}
Solution: {solution}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""

client, settings = get_client()

---
## BUILDING THE DATASET
---

In [ ]:
# An evaluation dataset contains inputs that we'll feed into our prompt.
# For each combination of prompt and input, we'll run the prompt and analyze the results.
# We can either create this dataset by hand or generate it automatically using Claude.


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(
    messages: list[MessageParam],
    *,
    system: str | None = None,
    stop_sequences: list[str] | None = None,
) -> str:
    params = {
        "model": settings.claude_model,
        "max_tokens": settings.default_max_tokens,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response: Message = client.messages.create(**params)
    text = "".join(block.text for block in response.content if isinstance(block, TextBlock))
    return text


# For a production-grade project, the cleaner pattern is a small function. It's explicit,
# testable, and easy to reuse across notebooks and scripts.
def build_prompt(task: str) -> str:
    """Fill the prompt template with a task description."""
    return PROMPT_TEMPLATE.format(task=task)


def generate_dataset() -> dict[str, str]:
    prompt = """
        Generate an evaluation dataset for a prompt evaluation. The dataset will be used
        to evaluate prompts that generate Python, JSON, or Regex specifically for
        AWS-related tasks. Generate an array of JSON objects, each representing task
        that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
          \\{
            "task": "Description of task",
            "format": "Python, JSON or Regex",
          \\},
          ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function,
        a single JSON object, or a single regex
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
        """

    messages: list[MessageParam] = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [ ]:
dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

In [ ]:
nb_data_dir = Path("data").resolve()
nb_data_dir.mkdir(parents=True, exist_ok=True)
file_path = nb_data_dir / "dataset.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print(f"Dataset JSON file created at: {file_path}")

---
## OUTPUT GRADERS
---

In [ ]:
def validate_json(text: str) -> int:
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text: str) -> int:
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text: str) -> int:
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [ ]:
def grade_by_model(test_case: dict[str, str], output: str) -> dict[str, str]:
    messages: list[MessageParam] = []
    eval_prompt = EVAL_PROMPT_TEMPLATE.format(task=test_case, solution=output)

    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

---
## RUNNING THE EVAL
---

The evaluation process follows a clear workflow: we take our dataset of test cases, combine each
one with our prompt template, send it to Claude for processing, and then evaluate the output using
a grader system.

In [ ]:
# This function takes a test case and merges it with our prompt template:
def run_prompt(test_case: dict[str, str]) -> str:
    """Merges the prompt and test case input, then returns the result"""
    prompt = PROMPT_TEMPLATE.format(task=test_case["task"])

    messages: list[MessageParam] = []
    add_user_message(messages, prompt)
    return chat(messages)


# This function orchestrates running a single test case and grading the result:
def run_test_case(test_case: dict[str, str]) -> TestOut:
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": int(score),
        "reasoning": reasoning,
    }


# This function coordinates the entire evaluation process:
def run_eval(dataset: list[dict[str, str]]) -> list[TestOut]:
    """Loads the dataset and calls run_test_case with each case"""
    results: list[TestOut] = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
with open(file_path, encoding="utf-8") as f:
    dataset = json.load(f)

results = run_eval(dataset)

# The evaluation returns a structured JSON array where each object represents one test case result:
print(json.dumps(results, indent=2))